In [0]:
# Setup paths and read Silver transactions

from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_transactions_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/transactions"
silver_accounts_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/accounts"

df_transactions = spark.read.format("delta").load(silver_transactions_path)
print(f"Total transactions: {df_transactions.count():,}")
df_transactions.printSchema()
df_transactions.select("account_id", "transaction_date", "balance").show(10)

Total transactions: 116,162
root
 |-- account_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- balance: double (nullable = true)

+----------+----------------+----------------+
|account_id|transaction_date|         balance|
+----------+----------------+----------------+
|  1196428'|      2019-03-05|-1.68284755405E9|
|  1196428'|      2019-03-05|-1.68794226613E9|
|  1196428'|      2019-03-05|-1.68722433183E9|
|  1196428'|      2019-03-05|-1.68984755405E9|
|  1196428'|      2019-03-05|-1.68034755405E9|
|  1196428'|      2019-03-05|-1.68654248413E9|
|  1196428'|      2019-03-05|-1.68692433183E9|
|  1196428'|      2019-03-05|-1.67984755405E9|
|  1196428'|      2019-03-05|-1.68384755405E9|
|  1196428'|      2019-03-05|-1.68746741683E9|
+----------+----------------+----------------+
only showing top 10 rows


In [0]:
# Use Window function to get latest balance per account

window_spec = Window.partitionBy("account_id").orderBy(col("transaction_date").desc())

df_latest = df_transactions.withColumn("row_num", row_number().over(window_spec))

df_latest.select("account_id", "transaction_date", "balance", "row_num").show(20)

+----------+----------------+----------------+-------+
|account_id|transaction_date|         balance|row_num|
+----------+----------------+----------------+-------+
|  1196428'|      2019-03-05|-1.68284755405E9|      1|
|  1196428'|      2019-03-05|-1.68794226613E9|      2|
|  1196428'|      2019-03-05|-1.68722433183E9|      3|
|  1196428'|      2019-03-05|-1.68984755405E9|      4|
|  1196428'|      2019-03-05|-1.68034755405E9|      5|
|  1196428'|      2019-03-05|-1.68654248413E9|      6|
|  1196428'|      2019-03-05|-1.68692433183E9|      7|
|  1196428'|      2019-03-05|-1.67984755405E9|      8|
|  1196428'|      2019-03-05|-1.68384755405E9|      9|
|  1196428'|      2019-03-05|-1.68746741683E9|     10|
|  1196428'|      2019-03-05|-1.67884755405E9|     11|
|  1196428'|      2019-03-05|-1.68597414205E9|     12|
|  1196428'|      2019-03-05|-1.68974755405E9|     13|
|  1196428'|      2019-03-05|-1.68730395386E9|     14|
|  1196428'|      2019-03-05|-1.68134755405E9|     15|
|  1196428

In [0]:
# Keep only the latest record for each account

df_accounts = df_latest.filter(col("row_num") == 1)

print(f"Total accounts: {df_accounts.count():,}")
df_accounts.select("account_id", "transaction_date", "balance").show(10)

Total accounts: 10
+-------------+----------------+----------------+
|   account_id|transaction_date|         balance|
+-------------+----------------+----------------+
|     1196428'|      2019-03-05|-1.68284755405E9|
|     1196711'|      2019-02-28|-1.58691558925E9|
|409000362497'|      2019-03-05|-1.90190209261E9|
|409000405747'|      2019-03-02| -5.4826745865E8|
|409000425051'|      2019-03-02| -3.5673478865E8|
|409000438611'|      2019-03-05| -5.4821049986E8|
|409000438620'|      2019-03-05| -5.3945395387E8|
|409000493201'|      2019-03-05|       743583.32|
|409000493210'|      2019-03-05| -5.4624972234E8|
|409000611074'|      2019-02-08|        462200.0|
+-------------+----------------+----------------+



In [0]:
# Rename balance and add account_type and branch_id columns

from pyspark.sql.functions import hash, abs

df_accounts = df_accounts.select(
    col("account_id"),
    col("balance").alias("latest_balance")
).withColumn("account_type", lit("Savings"))

# Use hash of account_id to randomly assign branch
branch_num = abs(hash(col("account_id"))) % 4

df_accounts = df_accounts.withColumn(
    "branch_id",
    when(branch_num == 0, lit("B1"))
    .when(branch_num == 1, lit("B2"))
    .when(branch_num == 2, lit("B3"))
    .otherwise(lit("B4"))
)

df_accounts.show(10, truncate=False)

+-------------+----------------+------------+---------+
|account_id   |latest_balance  |account_type|branch_id|
+-------------+----------------+------------+---------+
|1196428'     |-1.68284755405E9|Savings     |B4       |
|1196711'     |-1.58691558925E9|Savings     |B1       |
|409000362497'|-1.90190209261E9|Savings     |B1       |
|409000405747'|-5.4826745865E8 |Savings     |B2       |
|409000425051'|-3.5673478865E8 |Savings     |B3       |
|409000438611'|-5.4821049986E8 |Savings     |B1       |
|409000438620'|-5.3945395387E8 |Savings     |B3       |
|409000493201'|743583.32       |Savings     |B4       |
|409000493210'|-5.4624972234E8 |Savings     |B2       |
|409000611074'|462200.0        |Savings     |B1       |
+-------------+----------------+------------+---------+



In [0]:
# Ensure no duplicates and select final columns

print(f"Before deduplication: {df_accounts.count():,}")
df_accounts = df_accounts.dropDuplicates(["account_id"])
print(f"After deduplication: {df_accounts.count():,}")

df_accounts = df_accounts.select("account_id", "latest_balance", "account_type", "branch_id")
df_accounts.printSchema()

Before deduplication: 10
After deduplication: 10
root
 |-- account_id: string (nullable = true)
 |-- latest_balance: double (nullable = true)
 |-- account_type: string (nullable = false)
 |-- branch_id: string (nullable = false)



In [0]:
# Save as Delta table with schema overwrite

df_accounts.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_accounts_path)
print(f"✓ Saved to: {silver_accounts_path}")

✓ Saved to: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/accounts


In [0]:
# Show sample data ordered by account_id

df_accounts.orderBy("account_id").show(10, truncate=False)

+-------------+----------------+------------+---------+
|account_id   |latest_balance  |account_type|branch_id|
+-------------+----------------+------------+---------+
|1196428'     |-1.68284755405E9|Savings     |B4       |
|1196711'     |-1.58691558925E9|Savings     |B1       |
|409000362497'|-1.90190209261E9|Savings     |B1       |
|409000405747'|-5.4826745865E8 |Savings     |B2       |
|409000425051'|-3.5673478865E8 |Savings     |B3       |
|409000438611'|-5.4821049986E8 |Savings     |B1       |
|409000438620'|-5.3945395387E8 |Savings     |B3       |
|409000493201'|743583.32       |Savings     |B4       |
|409000493210'|-5.4624972234E8 |Savings     |B2       |
|409000611074'|462200.0        |Savings     |B1       |
+-------------+----------------+------------+---------+



In [0]:
# Display balance statistics

print("Balance Statistics:")
df_accounts.select(
    min("latest_balance").alias("min_balance"),
    max("latest_balance").alias("max_balance"),
    avg("latest_balance").alias("avg_balance")
).show()

print("\nAccount Type Distribution:")
df_accounts.groupBy("account_type").count().show()

Balance Statistics:
+----------------+-----------+----------------+
|     min_balance|max_balance|     avg_balance|
+----------------+-----------+----------------+
|-1.90190209261E9|  743583.32|-7.70937587596E8|
+----------------+-----------+----------------+


Account Type Distribution:
+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|   10|
+------------+-----+



In [0]:
# Verify branch assignment distribution

print("Accounts per Branch:")
df_accounts.groupBy("branch_id").count().orderBy("branch_id").show()

Accounts per Branch:
+---------+-----+
|branch_id|count|
+---------+-----+
|       B1|    4|
|       B2|    2|
|       B3|    2|
|       B4|    2|
+---------+-----+

